# Prep ParlaSpeech — chapter 1 (variant c)

Parse a ParlaSpeech-{LANG} JSONL into canonical pipeline JSONLs, convert the
per-utterance FLACs to 16 kHz mono WAVs, and emit **one file per instance-shape**
(not per task — multiple label keys live in one file).

**Recipes emitted here** (both whole-utterance, no slicing):
- `utterance_instance` — one record per utterance, scalar labels:
  `speaker_gender`, `filled_pause_present`, `filled_pause_count`,
  `sentiment_logit`, `sentiment_6`. The trainer picks one `label_key`.
- `utterance_frame` — one record per utterance, a 50 Hz `filled_pause` label
  sequence (where in time the FPs are).

Both share the same WAVs and the same speaker-grouped splits, so nothing leaks
and the splits agree across flavors.

**Inputs**
- Annotations: `data/unpacked/ParlaSpeech-{LANG}/.../ParlaSpeech-{LANG}.v3.0.jsonl`
- Audio: `data/unpacked/ParlaSpeech-{LANG}-audio/` — FLACs nested under `*.part*/`
  trees that differ by corpus (HR/RS: `partX/{hash}/`, CZ: `partX/audio/psp/Y/M/D/`).
  The splitter's basename index handles all layouts; downstream only sees our
  clean `data/cut_audio/ParlaSpeech-{LANG}/{hash}/{stem}.wav`.

**Not done here** (future recipes, registry stubs below): `event_instance`
(FP-quality, needs the annotator deliverable, `cut=True`) and `word_frame`
(primary stress, HR/RS only, `cut=True`).

---

## 0. Setup

In [5]:
import sys
from pathlib import Path

HERE = Path.cwd()
if HERE.name != "1_data_prep":
    candidate = HERE / "1_data_prep"
    if candidate.exists():
        HERE = candidate
sys.path.insert(0, str(HERE))

import utils_dataprep as udp
import utils_audio_splitter as uas

PROJECT_ROOT = udp.PROJECT_ROOT
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = /cache/ivanp/projects/slavic-speech-pipeline


Standard imports.

In [6]:
import json
from collections import Counter
from dataclasses import dataclass, field

from tqdm.auto import tqdm

---

## 1. Config

Key switches:
- `lang` — HR / RS / PL / CZ (auto-detected from `data/unpacked/` if left `""`).
- `recipes` — which instance-shapes to emit this run.
- `convert_audio` — FLAC → 16 kHz mono WAV (whole-file). `False` re-writes the
  JSONLs against already-converted WAVs without touching audio.
- `num_workers` — parallel FLAC→WAV conversion (each FLAC is independent, so
  threads win). 8 is a sane default.
- `audio_index_cache` — persistent `{basename → path}` index; built once,
  reused on later runs. Delete the file to force a rescan.

In [7]:
KNOWN_LANGS = ["HR", "RS", "PL", "CZ"]

@dataclass
class Config:
    # Language. "" → process every ParlaSpeech-{LANG} found under data/unpacked/.
    # Set explicitly (e.g. "RS") to pin one.
    lang: str = ""

    recipes: tuple = ("utterance_instance", "utterance_frame")

    jsonl_path:        str = ""
    audio_base_dir:    str = ""
    output_dir:        str = "data/processed_jsonl"
    output_wav_dir:    str = ""
    audio_index_cache: str = ""

    convert_audio: bool = True
    num_workers:   int  = 8
    cache_index:   bool = True

    frame_rate_hz: int = 50

    split_ratios: tuple = (0.8, 0.1, 0.1)
    split_seed:   str   = "parlaspeech-v1"
    split_group:  str   = "speaker"

    min_duration_s: float = 0.1

    test_mode:      bool = False                                       ############ TEST MODE
    test_n_records: int  = 500

cfg = Config()

# ── Resolve the set of languages to process ───────────────────────────────────
if cfg.lang:
    LANGS = [cfg.lang]
else:
    LANGS = [l for l in KNOWN_LANGS
             if (PROJECT_ROOT / f"data/unpacked/ParlaSpeech-{l}").exists()]
    if not LANGS:
        raise FileNotFoundError(
            "cfg.lang empty and no ParlaSpeech-{LANG} dirs under data/unpacked/. "
            "Run 10_download_data.ipynb first.")
    print(f"auto-detected langs: {LANGS}")

# ── Per-lang derivation (call once per lang; the work cells loop over LANGS) ───
def configure(lang: str):
    """Return the per-lang config view: (cfg-with-paths, L, l, DATASET, OUT_DIR, WAV_DIR)."""
    from dataclasses import replace
    L, l = lang, lang.lower()
    c = replace(
        cfg,
        lang=L,
        jsonl_path=f"data/unpacked/ParlaSpeech-{L}/ParlaSpeech-{L}.v3.0/ParlaSpeech-{L}.v3.0.jsonl",
        audio_base_dir=f"data/unpacked/ParlaSpeech-{L}-audio",
        output_wav_dir=f"data/cut_audio/ParlaSpeech-{L}",
        audio_index_cache=f"data/processed_jsonl/parlaspeech_{l}_audio_index.json",
    )
    DATASET = f"ParlaSpeech-{L}"
    OUT_DIR = c.output_dir if not c.test_mode else "data/test_processed_jsonl"
    WAV_DIR = c.output_wav_dir if not c.test_mode else f"data/cut_audio/test/ParlaSpeech-{L}"
    return c, L, l, DATASET, OUT_DIR, WAV_DIR

if cfg.test_mode:
    udp.banner("🧪 TEST MODE", char="-")

# Active lang for this pass = the first. (Wrap the work cells in `for LANG in LANGS:`
# and call configure(LANG) at the top of the loop to actually process all of them.)
cfg, L, l, DATASET, OUT_DIR, WAV_DIR = configure(LANGS[0])
print(f"active: {DATASET}  |  to process all {len(LANGS)}: loop the work cells over LANGS")
print(cfg)
print(f"dataset={DATASET}  out_dir={OUT_DIR}  wav_dir={WAV_DIR}")

auto-detected langs: ['HR', 'RS', 'PL', 'CZ']
active: ParlaSpeech-HR  |  to process all 4: loop the work cells over LANGS
Config(lang='HR', recipes=('utterance_instance', 'utterance_frame'), jsonl_path='data/unpacked/ParlaSpeech-HR/ParlaSpeech-HR.v3.0/ParlaSpeech-HR.v3.0.jsonl', audio_base_dir='data/unpacked/ParlaSpeech-HR-audio', output_dir='data/processed_jsonl', output_wav_dir='data/cut_audio/ParlaSpeech-HR', audio_index_cache='data/processed_jsonl/parlaspeech_hr_audio_index.json', convert_audio=True, num_workers=8, cache_index=True, frame_rate_hz=50, split_ratios=(0.8, 0.1, 0.1), split_seed='parlaspeech-v1', split_group='speaker', min_duration_s=0.1, test_mode=False, test_n_records=500)
dataset=ParlaSpeech-HR  out_dir=data/processed_jsonl  wav_dir=data/cut_audio/ParlaSpeech-HR


---

## 1.5 Recipe registry

A *recipe* is an instance-shape: `(unit × cut × instance|frame)`. Multiple label
keys can live in one recipe — task type is a downstream parameter, not a reason
to split files. This mirrors the `TARGETS` dict in `30_train_instance.ipynb`:
prep has recipes, the trainer has targets.

`requires` gates corpus-specific tiers (e.g. `primary_stress` is HR/RS only) so a
recipe fails loudly on a corpus that lacks them rather than emitting garbage.

In [8]:
RECIPES = {
    # ---- emitted by this notebook (whole utterance, no cut) -----------------
    "utterance_instance": dict(
        unit="utterance", level="instance", cut=False, requires=(),
        out=f"{OUT_DIR}/parlaspeech_{l}_utterance_instance.jsonl",
    ),
    "utterance_frame": dict(
        unit="utterance", level="frame", cut=False, requires=("filled_pauses",),
        out=f"{OUT_DIR}/parlaspeech_{l}_utterance_frame.jsonl",
    ),

    # ---- future (need cut=True via utils_audio_splitter) --------------------
    # "event_instance": one record per filled-pause event; label = FP quality
    #   (vowel / vowel+nasal / nasal / other / NA). Needs the annotator
    #   deliverable. unit="event", cut=True.
    # "word_frame": one record per word; 50 Hz primary-stress sequence.
    #   HR/RS only. unit="word", cut=True, requires=("primary_stress","words_align").
}

for name in cfg.recipes:
    if name not in RECIPES:
        raise ValueError(f"unknown recipe {name!r}. Known: {sorted(RECIPES)}")
print("emitting recipes:", list(cfg.recipes))

emitting recipes: ['utterance_instance', 'utterance_frame']


---

## 2. Locate the JSONL

In [9]:
jsonl_path = PROJECT_ROOT / cfg.jsonl_path
if not jsonl_path.exists():
    raise FileNotFoundError(
        f"JSONL not found: {jsonl_path}\n"
        f"Run 10_download_data.ipynb with datasets=['ParlaSpeech-{cfg.lang}'] first.")
print(f"✅ {jsonl_path.relative_to(PROJECT_ROOT)}  ({jsonl_path.stat().st_size/1e6:.0f} MB)")

✅ data/unpacked/ParlaSpeech-HR/ParlaSpeech-HR.v3.0/ParlaSpeech-HR.v3.0.jsonl  (15771 MB)


---

## 3. Preflight — peek at one record

In [10]:
with open(jsonl_path, encoding="utf-8") as f:
    _ex = json.loads(f.readline())
si = _ex.get("speaker_info", {})
print(f"id:            {_ex['id']}")
print(f"audio:         {_ex.get('audio')}")
print(f"audio_length:  {_ex.get('audio_length', 0):.3f}s")
print(f"text:          {str(_ex.get('text',''))[:70]}")
print(f"filled_pauses: {_ex.get('filled_pauses')}")
print(f"speaker:       {si.get('Speaker_ID','?')} / {si.get('Speaker_gender','?')} / {si.get('Lang','?')}")
print(f"has words:     {'words' in _ex}   has sentiment: {'sentiment' in _ex}")
print(f"has _aligns:   words_align={'words_align' in _ex}  chars_align={'chars_align' in _ex}  (HR/RS only)")

id:            ParlaMint-HR_2018-06-15-0.u86190_221-470
audio:         Pm2I_V_CTSQ/Pm2I_V_CTSQ_2428.94-2446.62.flac
audio_length:  17.680s
text:          S druge strane, prevladava također mišljenje, a i ovdje često razgovar
filled_pauses: [{'time_s': 14.34, 'time_e': 15.36, 'words_idx': 36}, {'time_s': 15.96, 'time_e': 16.14, 'words_idx': 37}]
speaker:       BabićAnte / M / Croatian
has words:     True   has sentiment: True
has _aligns:   words_align=True  chars_align=True  (HR/RS only)


---

## 4. Stream, parse, build canonical records

One canonical record per utterance, carrying **every** label key. `None` means
"not available for this utterance" (e.g. FP inference failed, or gender is `"-"`);
the trainer drops `None` per-task, so one failed tier never costs you the others.

We keep the `words` tier (needed for future word-level recipes). The two `_align`
tiers are large and HR/RS-only — extraction lines are present but commented out;
uncomment if you ever need them.

In [11]:
def gender_label(si):
    g = si.get("Speaker_gender")
    return g if g in ("M", "F") else None   # "-"/missing → None (dropped per-task)

def parse_record(r):
    """ParlaSpeech raw record → canonical record (or None if too short)."""
    dur = round(float(r.get("audio_length", 0.0)), 3)
    if dur < cfg.min_duration_s:
        return None

    si  = r.get("speaker_info", {})
    fps = r.get("filled_pauses")                 # None = inference failed; [] = none
    sent = r.get("sentiment") or {}
    raw_audio = r.get("audio")                   # "hash/hash_start-end.flac"
    file_hash = Path(raw_audio).parts[0] if raw_audio else None
    stem      = Path(raw_audio).stem if raw_audio else r["id"]

    return {
        "instance_id": r["id"],                  # ParlaMint id — globally unique
        "dataset":     DATASET,
        "file_id":     file_hash,                # source-video hash (provenance)
        "audio_path":  f"{WAV_DIR}/{file_hash}/{stem}.wav",
        "speaker":     si.get("Speaker_ID", "unknown"),
        "text":        r.get("text"),
        "labels": {
            "speaker_gender":       gender_label(si),
            "filled_pause_present": None if fps is None else int(bool(fps)),
            "filled_pause_count":   None if fps is None else len(fps),
            "sentiment_logit":      sent.get("ParlaSent_logit"),
            "sentiment_6":          sent.get("ParlaSent_6"),
        },
        "metadata": {
            "source_audio": raw_audio,           # resolver finds the FLAC by this basename
            "audio_length": dur,
            "lang":         si.get("Lang"),
            "speaker_info": si,                  # full metadata kept
            "words":        r.get("words"),      # kept — needed for word-level recipes
            # "words_align": r.get("words_align"),   # HR/RS only, bulky — enable if needed
            # "chars_align": r.get("chars_align"),   # HR/RS only, bulky — enable if needed
            "filled_pauses": fps,                # raw FP intervals (for the frame recipe)
        },
    }

records, n_total, n_short = [], 0, 0
with open(jsonl_path, encoding="utf-8") as f:
    for line in tqdm(f, desc=f"parsing {DATASET}", unit=" lines"):
        n_total += 1
        if cfg.test_mode and n_total > cfg.test_n_records:
            break
        rec = parse_record(json.loads(line))
        if rec is None:
            n_short += 1
            continue
        records.append(rec)

print(f"\nread {n_total} lines  kept {len(records)}  dropped(short) {n_short}")

parsing ParlaSpeech-HR: 0 lines [00:00, ? lines/s]


read 922679 lines  kept 922676  dropped(short) 3


---

## 5. Stats

In [12]:
gender = Counter(r["labels"]["speaker_gender"] for r in records)
fp_known = [r for r in records if r["labels"]["filled_pause_present"] is not None]
n_fp_pos = sum(r["labels"]["filled_pause_present"] for r in fp_known)
sent_known = sum(1 for r in records if r["labels"]["sentiment_logit"] is not None)
speakers = {r["speaker"] for r in records}

print(f"records:        {len(records)}")
print(f"speakers:       {len(speakers)} unique")
print(f"gender:         {dict(gender)}")
print(f"FP labelled:    {len(fp_known)}  (FP inference failed on {len(records)-len(fp_known)})")
if fp_known:
    print(f"  FP present:   {n_fp_pos} ({100*n_fp_pos/len(fp_known):.1f}%)")
print(f"sentiment:      {sent_known} with a logit")

records:        922676
speakers:       494 unique
gender:         {'M': 698542, 'F': 215000, None: 9134}
FP labelled:    881235  (FP inference failed on 41441)
  FP present:   188179 (21.4%)
sentiment:      922676 with a logit


---

## 6. Assign splits — grouped by speaker

Deterministic; the same speaker always lands in the same split, so no speaker
leaks between train/dev/test.

In [13]:
udp.assign_splits(records, ratios=cfg.split_ratios,
                  group_key=cfg.split_group, seed=cfg.split_seed, overwrite=True)
print("split distribution:", udp.split_summary(records))

split distribution: {'train': 754537, 'dev': 77407, 'test': 90732, 'other': 0}


---

## 7. Convert FLAC → WAV

Whole-file convert via `utils_audio_splitter`. The basename-index resolver finds
each FLAC regardless of `partX/` nesting; the index is cached so re-runs skip the
scan. Records whose FLAC can't be resolved are dropped (their `audio_path` would
point at a file that never gets written).

In [14]:
if cfg.convert_audio:
    resolver = uas.make_flac_index_resolver(
        cfg.audio_base_dir,
        record_key_path=("metadata", "source_audio"),
        index_cache_path=(cfg.audio_index_cache if cfg.cache_index else None),
    )
    records, stats = uas.cut_dataset(records, resolver, num_workers=cfg.num_workers)
    udp.banner("conversion stats", char="-")
    for k, v in stats.items():
        print(f"  {k:>18}: {v}")
else:
    print("convert_audio=False — assuming WAVs already exist; keeping all records")

scanned 922,679 files under data/unpacked/ParlaSpeech-HR-audio
  cached index → data/processed_jsonl/parlaspeech_hr_audio_index.json

----------------------------------------------------------------------
conversion stats
----------------------------------------------------------------------
               input: 922676
      missing_source: 0
          cut_failed: 0
    skipped_existing: 0
                kept: 922676
        short_warned: 0


---

## 8. Frame label helper

50 Hz binary sequence from the `filled_pauses` intervals.

In [15]:
def compute_frame_labels(filled_pauses, dur, hz):
    n = round(dur * hz)
    labels = [0] * n
    for fp in (filled_pauses or []):
        s = max(0, round(fp["time_s"] * hz))
        e = min(n, round(fp["time_e"] * hz))
        for i in range(s, e):
            labels[i] = 1
    return labels

---

## 9. Write the recipe JSONLs

`utterance_instance` carries the scalar labels. `utterance_frame` carries the
50 Hz `filled_pause` sequence and only includes utterances where FP inference
succeeded (a `None` FP tier can't become a label sequence).

In [16]:
def build_instance(r):
    out = {k: r[k] for k in ("instance_id", "dataset", "file_id",
                             "audio_path", "speaker", "text", "split")}
    out["labels"]   = dict(r["labels"])
    out["metadata"] = {k: v for k, v in r["metadata"].items() if k != "filled_pauses"}
    return out

def build_frame(r):
    fps = r["metadata"]["filled_pauses"]
    dur = r["metadata"]["audio_length"]
    out = {k: r[k] for k in ("instance_id", "dataset", "file_id",
                             "audio_path", "speaker", "text", "split")}
    out["frame_rate_hz"] = cfg.frame_rate_hz
    out["labels"] = {"filled_pause": compute_frame_labels(fps, dur, cfg.frame_rate_hz)}
    out["metadata"] = {k: v for k, v in r["metadata"].items() if k != "filled_pauses"}
    return out

written = {}
for name in cfg.recipes:
    spec = RECIPES[name]
    if spec["level"] == "instance":
        rows = [build_instance(r) for r in records]
    else:  # frame
        rows = [build_frame(r) for r in records
                if r["metadata"]["filled_pauses"] is not None]
    n = udp.write_jsonl(rows, spec["out"])
    written[name] = (spec["out"], n)
    print(f"✅ {name}: wrote {n} → {spec['out']}")

✅ utterance_instance: wrote 922676 → data/processed_jsonl/parlaspeech_hr_utterance_instance.jsonl
✅ utterance_frame: wrote 881235 → data/processed_jsonl/parlaspeech_hr_utterance_frame.jsonl


---

## 10. Sanity checks

In [17]:
for name, (path, n) in written.items():
    rows = udp.read_jsonl(path)
    n_tot, n_valid, errs = udp.validate_jsonl(rows)
    tag = "✅" if not errs else "⚠️ "
    print(f"{tag} {name}: {n_valid}/{n_tot} valid")
    for e in errs[:3]:
        print(f"     {e}")

    if RECIPES[name]["level"] == "frame":
        bad = 0
        for r in rows:
            exp = round((r["metadata"]["audio_length"]) * cfg.frame_rate_hz)
            if abs(len(r["labels"]["filled_pause"]) - exp) > 1:
                bad += 1
        print(f"     frame-length vs duration: {bad} mismatches (>1 frame)")
    else:
        present = Counter(r["labels"]["filled_pause_present"] for r in rows)
        gender  = Counter(r["labels"]["speaker_gender"] for r in rows)
        null_wav = sum(1 for r in rows if not r.get("audio_path"))
        print(f"     filled_pause_present: {dict(present)}")
        print(f"     speaker_gender:       {dict(gender)}")
        print(f"     audio_path null:      {null_wav}")

✅ utterance_instance: 922676/922676 valid
     filled_pause_present: {0: 693056, 1: 188179, None: 41441}
     speaker_gender:       {'M': 698542, 'F': 215000, None: 9134}
     audio_path null:      0
✅ utterance_frame: 881235/881235 valid
     frame-length vs duration: 0 mismatches (>1 frame)


---

## Next

- **Chapter 2** — point `20_sniff_dataset.ipynb` at either JSONL.
- **Chapter 3** — add a target to `30_train_instance.ipynb`'s `TARGETS`:
  `parlaspeech_{lang}_utterance_instance.jsonl` with `label_key` ∈
  `{speaker_gender, filled_pause_present, filled_pause_count}` (classification)
  or `sentiment_logit` (regression).
- **Chapter 4** — point the frame trainer at `..._utterance_frame.jsonl`.
- Other languages: change `cfg.lang`, re-run. The audio index for each lang is
  cached separately.